In [1]:
import sys
from IPython.display import display, Javascript

def restart_kernel():
    """Restart the Jupyter Notebook kernel to reflect changes in modules and packages."""
    display(Javascript("Jupyter.notebook.kernel.restart()"))
    print("Kernel is restarting...")

restart_kernel()

import bptf
from bptf import BPTF
import numpy as np
import pandas as pd
import sparse
import os
import shutil
from tqdm import tqdm
import pickle
import scipy.stats as st
import matplotlib.pyplot as plt
import torch
import tensorly
import cupy
import pickle

import multiprocessing
from joblib import Parallel, delayed
from tqdm.contrib.concurrent import process_map

print(os.getcwd())

import gc
gc.collect()

<IPython.core.display.Javascript object>

Kernel is restarting...


/home/luiyusen/projects/bptf_new


20

In [2]:
with open('sptensor.pkl', 'rb') as f:
    Y = pickle.load(f)
print(Y.shape)

(280, 280, 20, 294, 3)


In [3]:
# need to enforce self-country actions = 0
mask = Y.coords[0] != Y.coords[1]
new_coords = Y.coords[:, mask].copy()
new_data = Y.data[mask].copy()
Y = sparse.COO(new_coords, new_data, shape=Y.shape)

In [4]:
# need to mask unobserved dates for ICEWS and TERRIER
# ICEWS is missing data from 2024 (1 year)
# TERRIER is missing data from 2019 onwards (5 years)
#   database  index
# 0    ICEWS      0
# 1    GDELT      1
# 2  TERRIER      2
# ICEWS 
I_range = np.arange(Y.shape[0])
A_range = np.arange(Y.shape[2])
T_range = np.arange(Y.shape[3])
D_range = np.arange(Y.shape[4])

ICEWS_masked_months = np.arange(Y.shape[3])[-12:]
coordinates = np.meshgrid(I_range, I_range, A_range, ICEWS_masked_months, np.zeros(1))
flattened_indices = [np.ravel(coords) for coords in coordinates]
flattened_indices = np.vstack(flattened_indices)
flattened_indices = flattened_indices.astype(np.int64)
ICEWS_mask = sparse.COO(coords=flattened_indices, data=np.zeros(flattened_indices.shape[1]), shape=Y.shape)

# TERRIER
TERRIER_masked_months = np.arange(Y.shape[3])[-5*12:]
coordinates = np.meshgrid(I_range, I_range, A_range, TERRIER_masked_months, np.ones(1)*2)
flattened_indices = [np.ravel(coords) for coords in coordinates]
flattened_indices = np.vstack(flattened_indices)
flattened_indices = flattened_indices.astype(np.int64)
TERRIER_mask = sparse.COO(coords=flattened_indices, data=np.zeros(flattened_indices.shape[1]), shape=Y.shape)

In [5]:
def check_mask(tensor):
    if not isinstance(tensor, np.ndarray):
        tensor = tensor.todense()
    assert ((tensor == 0) | (tensor == 1)).all()

check_mask(ICEWS_mask + TERRIER_mask)